# 6. Events: Construction & Properties

Notebook 1 introduced `peyes.create_events` briefly. This notebook covers the resulting `Event` objects in depth —
their properties, how to spot outliers, and how to convert back and forth between events and per-sample labels.
Later notebooks (event metrics, matching, alignment) all build on what's here.

In [1]:
import numpy as np
import peyes
import _helpers

d = _helpers.load_example_trial()
detector = peyes.create_detector("engbert", missing_value=np.nan, min_event_duration=4, pad_blinks_time=0)
labels, detection_info = detector.detect(
    t=d["t"], x=d["x"], y=d["y"], pixel_size_cm=d["pixel_size"], viewer_distance_cm=d["viewer_distance"],
)
events = peyes.create_events(
    labels=labels, t=d["t"], x=d["x"], y=d["y"], pupil=d["pupil"],
    pixel_size=d["pixel_size"], viewer_distance=d["viewer_distance"],
)
len(events)

62

## Properties of a single event

Every `Event` exposes timing (`start_time`, `end_time`, `duration`), spatial extent (`amplitude`, `azimuth`,
`distance`, `dispersion`, `center_pixel`, ...), and kinematics (`peak_velocity`, `median_velocity`,
`min_velocity`). `.velocities(unit=...)` returns the full per-sample velocity trace (`"px"`, `"deg"`, or `"rad"`):

In [2]:
saccade = next(e for e in events if e.label == peyes.parse_label("saccade"))
print(f"{saccade.label.name}: {saccade.duration:.1f}ms, amplitude={saccade.amplitude:.2f}deg, "
      f"peak_velocity={saccade.peak_velocity:.1f}deg/s, azimuth={saccade.azimuth:.1f}deg")
saccade.velocities(unit="deg")[:5]

SACCADE: 60.0ms, amplitude=7.79deg, peak_velocity=158.5deg/s, azimuth=178.5deg


array([         nan,  68.02280399, 128.6404363 , 127.93786425,
       120.06213582])

## Outlier detection

`is_outlier` / `get_outlier_reasons()` flag events whose duration falls outside the configured min/max for their
label, or whose pixel coordinates fall outside the configured screen bounds (see notebook 3 for how to configure
those bounds):

In [3]:
[(e.label.name, e.duration, e.is_outlier, e.get_outlier_reasons()) for e in events if e.is_outlier]

[('FIXATION', 4.003000000000043, True, ['min_duration']),
 ('SACCADE', 4.0, True, ['min_duration']),
 ('FIXATION', 5.998000000000047, True, ['min_duration']),
 ('FIXATION', 18.019999999999982, True, ['min_duration']),
 ('SACCADE', 5.998000000000047, True, ['min_duration']),
 ('FIXATION', 31.99000000000001, True, ['min_duration']),
 ('SACCADE', 3.99799999999982, True, ['min_duration']),
 ('FIXATION', 30.0, True, ['min_duration']),
 ('SACCADE', 6.015999999999849, True, ['min_duration']),
 ('FIXATION', 14.007000000000062, True, ['min_duration']),
 ('SACCADE', 4.000999999999976, True, ['min_duration']),
 ('FIXATION', 6.003000000000156, True, ['min_duration']),
 ('FIXATION', 3.9969999999998436, True, ['min_duration']),
 ('FIXATION', 4.005999999999858, True, ['min_duration']),
 ('SACCADE', 3.999000000000251, True, ['min_duration']),
 ('FIXATION', 3.9989999999997963, True, ['min_duration']),
 ('SACCADE', 4.010000000000218, True, ['min_duration']),
 ('FIXATION', 4.0090000000000146, True, ['min

## `.summary()` and `summarize_events`

`.summary()` returns one event's features as a `pandas.Series`; `peyes.summarize_events` stacks that over a whole
sequence into a `DataFrame` (used throughout this guide):

In [4]:
saccade.summary()

event_type                                               SACCADE
label                                                          2
start_time                                               212.047
end_time                                                 272.057
duration                                                   60.01
distance                                              241.146818
amplitude                                               7.788051
azimuth                                                178.51383
peak_velocity                                         158.498845
median_velocity                                        79.263708
min_velocity                                           19.203813
cumulative_distance                                   319.304302
cumulative_amplitude                                   10.300293
start_x                                                 522.7053
start_y                                                  416.103
end_x                    

In [5]:
peyes.summarize_events(events).head()

,event_type,label,start_time,end_time,duration,distance,amplitude,azimuth,peak_velocity,median_velocity,...,start_x,start_y,end_x,end_y,center_pixel,pixel_std,dispersion,ellipse_area,is_outlier,outlier_reasons
0,FIXATION,1,6.000,210.046,204.046,7.356845,0.237962,99.967557,42.993417,13.650236,...,522.4870,423.0247,521.2136,415.7789,"(520.6592281553399, 418.31843883495134)","(1.3781321854123456, 2.85209201717164)",0.522272,0.049609,False,[]
1,SACCADE,2,212.047,272.057,60.010,241.146818,7.788051,178.513830,158.498845,79.263708,...,522.7053,416.1030,281.6396,409.8487,"(347.81459677419355, 412.33902903225817)","(85.4484891508158, 3.520229042144439)",8.706070,2.715996,False,[]
2,FIXATION,1,274.055,496.099,222.044,40.102058,1.297074,6.302341,43.679851,13.020898,...,278.8346,410.6481,318.6943,406.2459,"(291.6303732142857, 410.28303482142854)","(12.000146907694374, 1.5809510192619272)",1.601033,0.265221,False,[]
3,SACCADE,2,498.103,522.111,24.008,127.840747,4.133306,12.796403,148.080598,111.109023,...,319.6409,405.4971,444.3065,377.1820,"(381.0220769230769, 388.2436153846154)","(48.09015486131078, 14.103464151043285)",5.139701,3.510692,False,[]
4,FIXATION,1,524.112,528.115,4.003,2.411057,0.077987,194.343443,26.145356,19.518542,...,445.7363,378.2661,443.4004,378.8634,"(444.44613333333336, 378.6637)","(0.9691638813373706, 0.2811466165544314)",0.094876,0.001146,True,[min_duration]


## Converting back: events to labels

`peyes.events_to_labels` is the inverse of `create_events` — it rasterizes events back into a per-sample label
array. It needs the original sampling rate (available from the detector as `.sr`, or your own knowledge of the
recording) and the number of samples to produce:

In [6]:
roundtrip_labels = peyes.events_to_labels(events, sampling_rate=detector.sr, min_num_samples=len(labels))
agreement = np.mean(np.array(roundtrip_labels) == np.array(labels))
print(f"{agreement:.1%} of samples match the original labels")

93.0% of samples match the original labels


The two don't match 100%: `create_detector(...).detect()` merges or drops chunks shorter than
`min_event_duration` before returning labels, and very short leftover gaps round differently when rasterized back —
expected, not a bug.

## What's next

**[7 Event Metrics](./7%20Event%20Metrics.ipynb)** — aggregate statistics (rates, transitions, distributions) over a
whole sequence of events.